# Silver Transformation - dimvendor

This notebook builds the `dimvendor` dataset in the Silver layer.



## Run Shared Libraries



In [ ]:
%run ../Misc/SharedLibraries

In [ ]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimvendor"

## Read Source Tables



In [ ]:
vendorDf   = spark.table("bronze.vendtable")

In [ ]:
vendorDf.printSchema()

## Build Silver Dataset



In [ ]:
dimVendorDf = vendorDf.filter(vendorDf.RecordId.isNotNull()
    ).select(
        vendorDf.VendId.alias("VendorId"),
        F.trim(vendorDf.VendorName).alias("VendorName"),
        F.trim(vendorDf.Address).alias("Address"),
        F.trim(vendorDf.City).alias("City"),
        F.trim(vendorDf.State).alias("State"),
        F.trim(vendorDf.Country).alias("Country"),
        F.trim(vendorDf.ZipCode).alias("ZipCode"),
        F.trim(vendorDf.Region).alias("Region"),
        F.when(vendorDf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(vendorDf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
        F.from_utc_timestamp(vendorDf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
        F.from_utc_timestamp(vendorDf.ValidFrom,'CST').alias("ValidFrom"),
        F.when(vendorDf.ValidTo.isNull(), F.lit("1900-01-01")).otherwise(vendorDf.ValidTo).cast("timestamp").alias("ValidTo"),
        vendorDf.Active,
        vendorDf.RecordId.alias("VendorRecordId"),
        F.trim(vendorDf.TaxId).alias("TaxId"),
        F.trim(vendorDf.CurrencyCode).alias("CurrencyCode")
    ).withColumn("VendorDiscount",
        F.when(F.col("Country") == "US", F.lit(0.01))
         .when(F.col("Country") == "UK", F.lit(0.006))
         .otherwise(F.lit(0.00))
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("VendorHashKey", F.xxhash64("VendorRecordId")
    )

display(dimVendorDf)

## Prepare Final DataFrame



In [ ]:
df_final = dimVendorDf

## Verify Source Schema



In [ ]:
saveDeltaTableToCatalog(df_final,"silver",Entity)

In [ ]:
dimVendorDf.printSchema()